# PeptideScreen — 01: Design

Creates a new run, designs peptide candidates, and builds per-candidate folders.

**Steps:**
1. Configure target (edit the cell below)
2. Fetch PDB + analyze interface
3. Extract backbone at target length
4. ProteinMPNN sequence design
5. Build 3D structures
6. Create per-candidate folders on Google Drive

**Output:** `Google Drive/PeptideScreen_Runs/{run_name}/candidates/{ID}/`

In [ ]:
# ============================================================
# CELL 1: Install + Clone
# ============================================================
%cd /content
!rm -rf NewPipeline_07-14
!git clone https://github.com/PMONESKIN/NewPipeline_07-14.git
%cd NewPipeline_07-14
!pip install -q pyyaml biopython requests numpy scipy rdkit PeptideBuilder torch
print("Ready.")

In [ ]:
# ============================================================
# CELL 2: Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/PeptideScreen_Runs'
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f"Runs will be saved to: {DRIVE_BASE}")

In [ ]:
# ============================================================
# CELL 3: Configure Target
# ============================================================
# Edit these values for your target.
# Only name, pdb_id, receptor_chain, ligand_chain are required.
# Everything else has defaults.

TARGET_NAME = "Nrf2/KEAP1"
PDB_ID = "2FLU"
RECEPTOR_CHAIN = "X"
LIGAND_CHAIN = "P"

# Optional — set to None to use defaults
FIXED_MOTIF = "ETGE"          # Lock these residues during design (None = redesign all)
DESIGN_LENGTH = 16             # Peptide length (None = use full chain from PDB)
MPNN_TEMPERATURES = [0.1, 0.2, 0.3]  # Sampling diversity
INITIAL_POOL = 50              # Sequences per temperature
SEED_SEQUENCES = []            # Known binders to include (e.g., ["LDEETGEFL"])
RUN_BLAST = False              # BLAST check (~30 sec per candidate)

# ── Don't edit below this line ──
import yaml, json
from datetime import datetime
from pathlib import Path

# Build config
config = {
    'project': {'name': 'PeptideScreen', 'version': '2.0'},
    'targets': [{
        'name': TARGET_NAME,
        'pdb_id': PDB_ID,
        'receptor_chain': RECEPTOR_CHAIN,
        'ligand_chain': LIGAND_CHAIN,
    }],
    'candidates': {
        'initial_pool': INITIAL_POOL,
        'final_shortlist': 10,
        'length': {'min': 5, 'max': 30},
        'filters': {'max_molecular_weight': 2500, 'flag_aggregation_prone': True},
    },
    'outputs': {
        'structures': 'data/structures/',
        'candidates': 'data/candidates/',
        'docking': 'data/docking/',
        'processed': 'data/processed/',
        'reports': 'outputs/reports/',
        'figures': 'outputs/figures/',
    },
}

if FIXED_MOTIF:
    config['targets'][0]['fixed_motif'] = FIXED_MOTIF
if DESIGN_LENGTH:
    config['targets'][0]['design_chain_length'] = DESIGN_LENGTH
if MPNN_TEMPERATURES:
    config['targets'][0]['mpnn_temperatures'] = MPNN_TEMPERATURES
if SEED_SEQUENCES:
    config['targets'][0]['seed_sequences'] = SEED_SEQUENCES

with open('config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Create run name
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M')
safe_target = TARGET_NAME.lower().replace('/', '-').replace(' ', '-')
RUN_NAME = f"{timestamp}_{safe_target}"
RUN_DIR = f"{DRIVE_BASE}/{RUN_NAME}"

os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(f"{RUN_DIR}/structures", exist_ok=True)
os.makedirs(f"{RUN_DIR}/interface", exist_ok=True)
os.makedirs(f"{RUN_DIR}/candidates", exist_ok=True)

# Save config to run directory
with open(f"{RUN_DIR}/config.yaml", 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Save run info
run_info = {
    'run_name': RUN_NAME,
    'created': datetime.now().isoformat(),
    'target': TARGET_NAME,
    'pdb_id': PDB_ID,
    'design_length': DESIGN_LENGTH,
    'fixed_motif': FIXED_MOTIF,
    'status': 'created',
}
with open(f"{RUN_DIR}/run_info.json", 'w') as f:
    json.dump(run_info, f, indent=2)

# Save RUN_DIR for other cells
with open('/content/current_run.txt', 'w') as f:
    f.write(RUN_DIR)

print(f"Run: {RUN_NAME}")
print(f"Directory: {RUN_DIR}")
print(f"Target: {TARGET_NAME} ({PDB_ID})")
print(f"Design length: {DESIGN_LENGTH}")
print(f"Fixed motif: {FIXED_MOTIF}")
print(f"Config saved.")

In [ ]:
# ============================================================
# CELL 4: Fetch PDB + Analyze Interface
# ============================================================
import requests
import numpy as np
from Bio.PDB import PDBParser, NeighborSearch
from Bio.PDB.Polypeptide import is_aa, protein_letters_3to1

with open('/content/current_run.txt') as f:
    RUN_DIR = f.read().strip()

# Download PDB
print(f"Downloading {PDB_ID}...")
url = f"https://files.rcsb.org/download/{PDB_ID.upper()}.pdb"
resp = requests.get(url, timeout=30)
pdb_path = f"{RUN_DIR}/structures/{PDB_ID.upper()}.pdb"
with open(pdb_path, 'w') as f:
    f.write(resp.text)
print(f"Saved: {pdb_path}")

# Analyze interface
parser = PDBParser(QUIET=True)
structure = parser.get_structure(PDB_ID, pdb_path)

receptor_residues = []
ligand_residues = []
for model in structure:
    for chain in model:
        if chain.id == RECEPTOR_CHAIN:
            receptor_residues = [r for r in chain if is_aa(r, standard=True)]
        elif chain.id == LIGAND_CHAIN:
            ligand_residues = [r for r in chain if is_aa(r, standard=True)]

print(f"Receptor (chain {RECEPTOR_CHAIN}): {len(receptor_residues)} residues")
print(f"Ligand (chain {LIGAND_CHAIN}): {len(ligand_residues)} residues")

# Auto-detect interface residues
receptor_atoms = [a for r in receptor_residues for a in r if a.element != 'H']
ns = NeighborSearch(receptor_atoms)
interface = set()
for res in ligand_residues:
    for atom in res:
        if atom.element == 'H':
            continue
        nearby = ns.search(atom.get_vector().get_array(), 4.5, level='R')
        for nr in nearby:
            if is_aa(nr, standard=True):
                interface.add(nr.get_id()[1])

active_residues = sorted(interface)
ligand_seq = ''.join(protein_letters_3to1.get(r.get_resname(), 'X') for r in ligand_residues)

# Binding box from ligand coordinates
coords = []
for res in ligand_residues:
    for atom in res:
        if atom.element != 'H':
            coords.append(atom.get_vector().get_array())
coords = np.array(coords)
center = coords.mean(axis=0)
span = coords.max(axis=0) - coords.min(axis=0)

# Check motif
motif_found = FIXED_MOTIF and FIXED_MOTIF in ligand_seq

# Save interface data
interface_data = {
    'target_name': TARGET_NAME,
    'pdb_id': PDB_ID.upper(),
    'pdb_path': pdb_path,
    'receptor_chain': RECEPTOR_CHAIN,
    'ligand_chain': LIGAND_CHAIN,
    'receptor_residue_count': len(receptor_residues),
    'ligand_residue_count': len(ligand_residues),
    'active_residues': active_residues,
    'ligand_sequence': ligand_seq,
    'design_chain_length': DESIGN_LENGTH or len(ligand_residues),
    'fixed_motif': FIXED_MOTIF if motif_found else None,
    'mpnn_temperatures': MPNN_TEMPERATURES,
    'seed_sequences': SEED_SEQUENCES,
    'binding_box': {
        'center_x': round(float(center[0]), 2),
        'center_y': round(float(center[1]), 2),
        'center_z': round(float(center[2]), 2),
    },
}

safe_name = TARGET_NAME.lower().replace('/', '_').replace(' ', '_')
interface_path = f"{RUN_DIR}/interface/{safe_name}_interface.json"
with open(interface_path, 'w') as f:
    json.dump(interface_data, f, indent=2)

print(f"\nInterface analysis:")
print(f"  Active residues: {len(active_residues)}")
print(f"  Ligand sequence: {ligand_seq}")
print(f"  Fixed motif '{FIXED_MOTIF}': {'FOUND' if motif_found else 'NOT FOUND'}")
print(f"  Saved: {interface_path}")

In [ ]:
# ============================================================
# CELL 5: Setup ProteinMPNN (one-time)
# ============================================================
import os
if not os.path.exists('tools/ProteinMPNN/protein_mpnn_run.py'):
    !python -u modules/02_design/setup_proteinmpnn.py
else:
    print("ProteinMPNN already set up.")

In [ ]:
# ============================================================
# CELL 6: ProteinMPNN Design + Build Structures
# ============================================================
import subprocess, sys, tempfile, shutil, re, io
from Bio.PDB import PDBParser, PDBIO, Select
from Bio.PDB.Polypeptide import is_aa, protein_letters_3to1
from PeptideBuilder import make_structure as make_peptide

with open('/content/current_run.txt') as f:
    RUN_DIR = f.read().strip()
with open(f"{RUN_DIR}/interface/{safe_name}_interface.json") as f:
    interface_data = json.load(f)

pdb_path = interface_data['pdb_path']
design_length = interface_data['design_chain_length']
fixed_motif = interface_data.get('fixed_motif')
temperatures = interface_data.get('mpnn_temperatures', [0.1, 0.2, 0.3])

MPNN_DIR = Path('tools/ProteinMPNN')

# Parse chains
mpnn_output = Path(f"{RUN_DIR}/mpnn_output")
mpnn_output.mkdir(parents=True, exist_ok=True)
parsed_dir = mpnn_output / 'parsed'
parsed_dir.mkdir(exist_ok=True)

with tempfile.TemporaryDirectory() as tmp:
    shutil.copy(pdb_path, f"{tmp}/{PDB_ID}.pdb")
    subprocess.run([sys.executable, str(MPNN_DIR / 'helper_scripts/parse_multiple_chains.py'),
                   '--input_path', tmp, '--output_path', str(parsed_dir / 'parsed.jsonl')],
                  capture_output=True, text=True, check=True)

subprocess.run([sys.executable, str(MPNN_DIR / 'helper_scripts/assign_fixed_chains.py'),
               '--input_path', str(parsed_dir / 'parsed.jsonl'),
               '--output_path', str(parsed_dir / 'assigned.jsonl'),
               '--chain_list', LIGAND_CHAIN],
              capture_output=True, text=True, check=True)

# Fixed motif positions
mpnn_cmd = [sys.executable, str(MPNN_DIR / 'protein_mpnn_run.py'),
            '--jsonl_path', str(parsed_dir / 'parsed.jsonl'),
            '--chain_id_jsonl', str(parsed_dir / 'assigned.jsonl'),
            '--out_folder', str(mpnn_output),
            '--num_seq_per_target', str(INITIAL_POOL),
            '--sampling_temp', ' '.join(str(t) for t in temperatures),
            '--seed', '42', '--batch_size', '1']

if fixed_motif:
    # Find motif positions in ligand chain
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure(PDB_ID, pdb_path)
    for model in struct:
        for chain in model:
            if chain.id == LIGAND_CHAIN:
                seq = ''.join(protein_letters_3to1.get(r.get_resname(), 'X')
                            for r in chain if is_aa(r, standard=True))
                idx = seq.find(fixed_motif)
                if idx >= 0:
                    positions = list(range(idx + 1, idx + 1 + len(fixed_motif)))
                    print(f"Locking motif '{fixed_motif}' at positions {positions}")
                    pos_dict = {PDB_ID.upper(): {LIGAND_CHAIN: positions, RECEPTOR_CHAIN: []}}
                    fixed_pos_file = parsed_dir / 'fixed_positions.jsonl'
                    with open(fixed_pos_file, 'w') as fp:
                        fp.write(json.dumps(pos_dict) + '\n')
                    mpnn_cmd.extend(['--fixed_positions_jsonl', str(fixed_pos_file)])

print(f"Running ProteinMPNN ({INITIAL_POOL} seqs x {len(temperatures)} temps)...")
result = subprocess.run(mpnn_cmd, capture_output=True, text=True)
if result.returncode != 0:
    print(f"GPU failed, retrying on CPU...")
    if '--use_gpu' in mpnn_cmd:
        mpnn_cmd.remove('--use_gpu')
    result = subprocess.run(mpnn_cmd, capture_output=True, text=True, check=True)

# Parse FASTA output
fasta_files = list((mpnn_output / 'seqs').glob('*.fa'))
if not fasta_files:
    raise FileNotFoundError("No FASTA output")

entries = []
with open(fasta_files[0]) as f:
    lines = f.read().strip().split('\n')
i = 0
while i < len(lines):
    if lines[i].startswith('>'):
        entries.append((lines[i][1:], lines[i+1].strip() if i+1 < len(lines) else ''))
        i += 2
    else:
        i += 1

# Parse into candidates, dedup
seen = set()
candidates = []
counter = 0

# Add seeds first
for seq in SEED_SEQUENCES:
    seq = seq.upper().strip()
    if seq not in seen and 5 <= len(seq) <= 30:
        seen.add(seq)
        counter += 1
        prefix = re.sub(r'[^A-Z0-9]', '', TARGET_NAME.upper())[:8]
        cid = f"{prefix}-{counter:03d}"
        candidates.append({'id': cid, 'sequence': seq, 'length': len(seq),
                          'target_name': TARGET_NAME, 'design_source': 'seed',
                          'mpnn_score': None})

# Add MPNN designs
ref_seq = entries[0][1] if entries else ''
for header, seq in entries[1:]:
    designed = seq[:design_length].upper()
    if designed in seen or len(designed) < 5 or len(designed) > 30:
        continue
    seen.add(designed)
    
    score = None
    for part in header.split(','):
        part = part.strip()
        if part.startswith('score='):
            try: score = float(part.split('=')[1])
            except: pass
    
    counter += 1
    prefix = re.sub(r'[^A-Z0-9]', '', TARGET_NAME.upper())[:8]
    cid = f"{prefix}-{counter:03d}"
    candidates.append({'id': cid, 'sequence': designed, 'length': len(designed),
                      'target_name': TARGET_NAME, 'design_source': 'proteinmpnn',
                      'mpnn_score': score})

print(f"\n{len(candidates)} unique candidates designed.")

# Build 3D structures and create per-candidate folders
print(f"\nBuilding structures and creating candidate folders...")

for c in candidates:
    cid = c['id']
    seq = c['sequence']
    
    # Create candidate folder
    cand_dir = f"{RUN_DIR}/candidates/{cid}"
    os.makedirs(cand_dir, exist_ok=True)
    
    # Build 3D structure
    n = len(seq)
    phi = [-180.0] * (n - 1)
    psi = [180.0] * (n - 1)
    struct = make_peptide(seq, phi, psi)
    
    pdb_io = PDBIO()
    pdb_io.set_structure(struct)
    pdb_io.save(f"{cand_dir}/structure.pdb")
    
    # Save candidate info
    info = {
        'id': cid,
        'sequence': seq,
        'length': len(seq),
        'target_name': TARGET_NAME,
        'design_source': c['design_source'],
        'mpnn_score': c['mpnn_score'],
        'fixed_motif': FIXED_MOTIF,
        'status': 'designed',
    }
    with open(f"{cand_dir}/info.json", 'w') as f:
        json.dump(info, f, indent=2)

# Save master candidate list
with open(f"{RUN_DIR}/candidates/candidate_list.json", 'w') as f:
    json.dump(candidates, f, indent=2)

# Update run info
run_info['status'] = 'designed'
run_info['n_candidates'] = len(candidates)
with open(f"{RUN_DIR}/run_info.json", 'w') as f:
    json.dump(run_info, f, indent=2)

print(f"\nDone. {len(candidates)} candidate folders created.")
print(f"\nFirst 5:")
for c in candidates[:5]:
    print(f"  {c['id']}  {c['sequence']}  MPNN={c.get('mpnn_score', 'N/A')}")

print(f"\nRun directory: {RUN_DIR}")

In [ ]:
# ============================================================
# CELL 7: BLAST Check (Optional)
# ============================================================
# Set RUN_BLAST = True in Cell 3 to enable.
# Takes ~30 seconds per candidate.

if RUN_BLAST:
    from Bio.Blast import NCBIWWW, NCBIXML
    import time
    
    with open('/content/current_run.txt') as f:
        RUN_DIR = f.read().strip()
    with open(f"{RUN_DIR}/candidates/candidate_list.json") as f:
        candidates = json.load(f)
    
    print(f"BLAST checking {len(candidates)} candidates...")
    
    for i, c in enumerate(candidates):
        cid = c['id']
        seq = c['sequence']
        cand_dir = f"{RUN_DIR}/candidates/{cid}"
        
        # Skip if already checked
        blast_path = f"{cand_dir}/blast.json"
        if os.path.exists(blast_path):
            continue
        
        print(f"  [{i+1}/{len(candidates)}] {cid} ({seq})")
        
        try:
            result_handle = NCBIWWW.qblast('blastp', 'swissprot', seq,
                                           hitlist_size=3, expect=10.0,
                                           word_size=2, matrix_name='PAM30',
                                           gapcosts='9 1')
            blast_record = next(NCBIXML.parse(result_handle))
            
            hits = []
            for alignment in blast_record.alignments[:3]:
                for hsp in alignment.hsps[:1]:
                    identity = hsp.identities / hsp.align_length * 100
                    hits.append({'title': alignment.title[:100],
                                'identity_pct': round(identity, 1),
                                'evalue': hsp.expect})
            
            blast_result = {
                'hits': hits,
                'best_identity': hits[0]['identity_pct'] if hits else 0,
                'known_match': hits[0]['identity_pct'] >= 95 if hits else False,
            }
        except Exception as e:
            blast_result = {'hits': [], 'best_identity': 0, 'error': str(e)}
        
        with open(blast_path, 'w') as f:
            json.dump(blast_result, f, indent=2)
        
        time.sleep(3)
    
    print("BLAST check complete.")
else:
    print("BLAST check skipped. Set RUN_BLAST = True in Cell 3 to enable.")

In [ ]:
# ============================================================
# CELL 8: Verify Results
# ============================================================
with open('/content/current_run.txt') as f:
    RUN_DIR = f.read().strip()

with open(f"{RUN_DIR}/candidates/candidate_list.json") as f:
    candidates = json.load(f)

print(f"{'='*60}")
print(f"  Design Complete: {RUN_DIR.split('/')[-1]}")
print(f"{'='*60}")
print(f"  Target: {TARGET_NAME}")
print(f"  PDB: {PDB_ID}")
print(f"  Design length: {DESIGN_LENGTH}")
print(f"  Fixed motif: {FIXED_MOTIF}")
print(f"  Candidates: {len(candidates)}")

# Verify folder structure
cand_dirs = [d for d in os.listdir(f"{RUN_DIR}/candidates")
             if os.path.isdir(f"{RUN_DIR}/candidates/{d}")]
print(f"  Candidate folders: {len(cand_dirs)}")

# Check a sample candidate
sample = candidates[0]
sample_dir = f"{RUN_DIR}/candidates/{sample['id']}"
sample_files = os.listdir(sample_dir)
print(f"\n  Sample folder ({sample['id']}):")
for f in sorted(sample_files):
    size = os.path.getsize(f"{sample_dir}/{f}")
    print(f"    {f} ({size} bytes)")

print(f"\n  All candidates:")
print(f"  {'ID':<16} {'Sequence':<22} {'MPNN':<10} {'Source'}")
print(f"  {'-'*16} {'-'*22} {'-'*10} {'-'*12}")
for c in candidates:
    seq = c['sequence'][:20] + '..' if len(c['sequence']) > 20 else c['sequence']
    mpnn = f"{c['mpnn_score']:.4f}" if c.get('mpnn_score') else 'N/A'
    print(f"  {c['id']:<16} {seq:<22} {mpnn:<10} {c['design_source']}")

print(f"\n{'='*60}")
print(f"  Next: Open 02_AF2_Filter.ipynb")
print(f"  Run directory: {RUN_DIR}")
print(f"{'='*60}")